# 00 · World Bank Indicators API  -  from zero to a real pull

**Goal.** Learn the World Bank Indicators API well enough to pull any
indicator × country × year slice you need for economic-development
research. We do three progressively richer examples, then hand off to
the reusable client in `src/wb_client.py`.

**Why this API?**
- Free, no key needed.
- ~1,600 indicators × 217+ countries+aggregates × back to 1960.
- Same JSON shape for every request  -  once you know the pattern, you're done.

**Docs to bookmark:**
- Basic call structure: <https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-basic-api-call-structure>
- Indicator search UI (where you find indicator codes): <https://data.worldbank.org/indicator>
- Country + aggregate list: <https://api.worldbank.org/v2/country?format=json>


## 1. The URL pattern

```
https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?format=json&date={start}:{end}&per_page=20000
```

- `country`  -  either `all`, an ISO3 code (e.g. `USA`), or a `;`-joined list.
- `indicator`  -  the WB code for the series (e.g. `NY.GDP.PCAP.KD` = GDP per capita, constant 2015 US$).
- `date`  -  a year range like `1990:2023`.
- `format=json`  -  otherwise you get XML.

**The one quirk:** every JSON response is a **2-element array**. Element 0
is pagination metadata; element 1 is the actual data rows. Every WB client
starts by unpacking that.


In [ ]:
import requests

url = ("https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.PCAP.KD"
       "?format=json&date=2000:2023&per_page=100")
payload = requests.get(url, timeout=30).json()

print("Type of payload:", type(payload))
print("Length:", len(payload))
print("Meta (element 0):", payload[0])
print("First data row (element 1[0]):", payload[1][0])


## 2. Flatten a single-indicator response into a DataFrame

In [ ]:
import pandas as pd

def to_df(payload):
    _, rows = payload
    return pd.DataFrame([
        {"iso3": r.get("countryiso3code"),
         "country": r["country"]["value"],
         "indicator_id": r["indicator"]["id"],
         "year": int(r["date"]),
         "value": r["value"]}
        for r in rows
    ])

usa = to_df(payload).sort_values("year")
usa.tail()


## 3. Pull many countries at once

The `country` slot accepts a semicolon-joined list. Handy for small
comparison studies. For a global panel we use `all`, but WB then also
returns *aggregates* (world, income groups, regions) that we filter
out later with the country metadata endpoint.


In [ ]:
url = ("https://api.worldbank.org/v2/country/USA;GBR;JPN;DEU;KOR/indicator/"
       "SP.DYN.LE00.IN?format=json&date=1990:2022&per_page=1000")
life = to_df(requests.get(url, timeout=30).json())
print(f"{len(life)} rows across {life['iso3'].nunique()} countries")
life.head()


## 4. Pagination

If your request returns more rows than fit on one page (max 20,000/page),
the metadata element gives you `pages > 1`. You fetch the next page with
`&page=2`, `&page=3`, etc. Our client does this transparently  -  but here's
the pattern by hand so you understand what's happening.


In [ ]:
def paged_fetch(country, indicator, date):
    rows, page = [], 1
    while True:
        u = (f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"
             f"?format=json&date={date}&per_page=20000&page={page}")
        meta, chunk = requests.get(u, timeout=30).json()
        rows.extend(chunk or [])
        if page >= meta["pages"]:
            break
        page += 1
    return rows

rows = paged_fetch("all", "NY.GDP.PCAP.KD", "1960:2023")
print(f"Total rows: {len(rows):,}   (about 217 countries × 64 years)")


## 5. Hand off to the reusable client

`src/wb_client.py` bundles everything we did above plus:
- parquet caching so re-running notebooks doesn't re-hit the API,
- a `fetch_indicators()` helper that pulls multiple indicators and pivots
  them into a wide country-year table,
- `fetch_country_metadata()` to get the region + income classification
  we need to drop aggregate rows.


In [ ]:
import sys
sys.path.insert(0, "..")

from src.wb_client import fetch_indicator, fetch_indicators, fetch_country_metadata

# Country classification (region + income group)
meta = fetch_country_metadata()
print("Total entries:", len(meta), "| Real countries:", (~meta["is_aggregate"]).sum())
meta.head()


In [ ]:
# Wide table for a mini panel: 3 indicators × all countries × 2000-2022
panel = fetch_indicators(
    indicators=["NY.GDP.PCAP.KD", "SP.DYN.LE00.IN", "EN.GHG.CO2.PC.CE.AR5"],
    countries="all",
    start=2000, end=2022,
)
# Drop aggregate rows
real_countries = set(meta.loc[~meta["is_aggregate"], "iso3"])
panel = panel[panel["iso3"].isin(real_countries)]
print(f"Panel: {len(panel):,} country-years × {panel.shape[1]} cols")
panel.head()


## 6. Cheat sheet  -  indicator codes we'll use

| Domain        | Code                     | What it is |
|--------------|--------------------------|--------|
| Growth       | `NY.GDP.PCAP.KD`         | GDP per capita, constant 2015 US$ |
| Growth       | `NY.GDP.PCAP.KD.ZG`      | GDP per capita growth, annual % |
| Institutions | `RL.EST`, `CC.EST`, `GE.EST` | Rule of law, corruption control, gov't effectiveness (WGI) |
| Health       | `SP.DYN.LE00.IN`         | Life expectancy at birth, total |
| Health       | `SP.DYN.LE00.FE.IN` / `MA.IN` | Life expectancy, female / male |
| Health       | `SH.XPD.CHEX.GD.ZS`      | Current health expenditure, % GDP |
| Environment  | `EN.GHG.CO2.PC.CE.AR5`   | CO2 emissions per capita (t) |
| Economy      | `NE.TRD.GNFS.ZS`         | Trade, % of GDP |
| Economy      | `NE.GDI.TOTL.ZS`         | Gross capital formation, % of GDP |
| Demographics | `SP.POP.TOTL`, `SP.URB.TOTL.IN.ZS` | Population, urban share |

You're ready. Open `01_rq1_institutions_growth.ipynb` next.
